# HoloMine Latent Domain Regime Model

Notebook ini membangun model harga properti dari teks dan structured features yang diekstrak langsung dari deskripsi. Fokus utama adalah perbedaan karakter pasar, perubahan proporsi rentang harga, dan kebutuhan specialist models untuk properti bernilai tinggi.

Pipeline terdiri dari sparse text regression, ordinal classification, soft routing, location priors, nearest-neighbor retrieval, domain-aware experts, dan residual LightGBM. Pemilihan model dilakukan menggunakan prediksi out-of-fold.


## Validasi dan pemilihan model

Semua kandidat dibandingkan menggunakan prediksi out-of-fold. Pemeriksaan utama meliputi MAE keseluruhan, MAE per rentang harga, bias prediksi, serta AUC pada beberapa batas harga. Kandidat dengan hasil paling konsisten digunakan untuk membuat submission.

File keluaran berisi submission, prediksi out-of-fold, dan ringkasan metrik agar hasil run dapat diperiksa tanpa melatih ulang model.


In [ ]:
from __future__ import annotations

import argparse
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack
import lightgbm as lgb
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import Ridge, SGDClassifier
from sklearn.metrics import mean_absolute_error, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors


SEED = 42
THRESHOLDS = np.array(
    [100_000, 300_000, 500_000, 800_000, 1_000_000, 2_000_000, 5_000_000, 10_000_000],
    dtype=float,
)
EDGES = np.r_[0.0, THRESHOLDS, np.inf]
NUMBER_WORDS = {
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "seven": 7,
    "eight": 8,
    "nine": 9,
    "ten": 10,
    "eleven": 11,
    "twelve": 12,
}
NUM_TOKEN = r"(?:\d+(?:\.\d+)?|" + "|".join(NUMBER_WORDS) + r")"


def clean_text(value: object) -> str:
    text = str(value).replace("[Redacted Entity]", " redactedentity ")
    text = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def as_number(value: str) -> float:
    value = value.lower().replace(",", "").strip()
    return float(NUMBER_WORDS.get(value, value))


def first_number(patterns: list[str], text: str) -> float:
    for pattern in patterns:
        match = re.search(pattern, text, re.I)
        if match:
            try:
                return as_number(match.group(1))
            except (TypeError, ValueError):
                continue
    return np.nan


def parse_property(value: object) -> dict[str, float | str]:
    text = clean_text(value)
    lower = text.lower()

    beds = first_number(
        [
            rf"\b({NUM_TOKEN})\s*[- ]?(?:bedrooms?|beds?|bdrms?|bdrm|bd|br)\b",
            rf"\b(?:bedrooms?|beds?|bdrms?|bdrm)\s*[:\-]?\s*({NUM_TOKEN})\b",
        ],
        lower,
    )
    baths = first_number(
        [
            rf"\b({NUM_TOKEN})\s*[- ]?(?:bathrooms?|baths?|ba)\b",
            rf"\b(?:bathrooms?|baths?)\s*[:\-]?\s*({NUM_TOKEN})\b",
        ],
        lower,
    )
    full_baths = first_number([rf"\b({NUM_TOKEN})\s*full\s*(?:bathrooms?|baths?)\b"], lower)
    half_baths = first_number([rf"\b({NUM_TOKEN})\s*half\s*(?:bathrooms?|baths?)\b"], lower)
    if np.isnan(baths) and (np.isfinite(full_baths) or np.isfinite(half_baths)):
        baths = (full_baths if np.isfinite(full_baths) else 0.0) + 0.5 * (
            half_baths if np.isfinite(half_baths) else 0.0
        )

    sqft = first_number(
        [
            r"\b([\d,]{3,7}(?:\.\d+)?)\s*(?:sq\.?\s*ft\.?|sqft|square\s*feet|square\s*foot|sf)\b",
            r"\b(?:living\s+area|interior|approximately|approx\.?)\D{0,20}([\d,]{3,7})\s*(?:sq\.?\s*ft\.?|sqft|sf|square\s*feet)?\b",
        ],
        lower,
    )
    match_k = re.search(r"\b(\d+(?:\.\d+)?)\s*k\s*(?:sq\.?\s*ft\.?|sqft|sf)\b", lower)
    if np.isnan(sqft) and match_k:
        sqft = float(match_k.group(1)) * 1_000.0

    acres = first_number([r"\b(\d+(?:\.\d+)?)\s*(?:acres?|acre)\b"], lower)
    lot_sqft = first_number(
        [r"\blot\D{0,18}([\d,]{3,9})\s*(?:sq\.?\s*ft\.?|sqft|square\s*feet|sf)\b"], lower
    )
    garage = first_number(
        [
            rf"\b({NUM_TOKEN})\s*[- ]?car\s+(?:attached\s+|detached\s+)?garage\b",
            rf"\bgarage\D{{0,12}}({NUM_TOKEN})\s*(?:cars?|vehicles?|stalls?)\b",
        ],
        lower,
    )
    year = first_number(
        [
            r"\b(?:built|constructed|year\s+built|completed)\D{0,15}((?:17|18|19|20)\d{2})\b",
            r"\b((?:17|18|19|20)\d{2})\s+(?:bungalow|cottage|colonial|ranch|home|house|residence)\b",
        ],
        lower,
    )
    stories = first_number(
        [rf"\b({NUM_TOKEN})\s*[- ]?(?:story|stories|storey|storeys)\b"], lower
    )

    bounds = {
        "beds": (beds, 0, 20),
        "baths": (baths, 0, 20),
        "sqft": (sqft, 150, 50_000),
        "acres": (acres, 0, 100_000),
        "lot_sqft": (lot_sqft, 100, 100_000_000),
        "garage": (garage, 0, 20),
        "year": (year, 1700, 2027),
        "stories": (stories, 0, 20),
    }
    numeric: dict[str, float] = {}
    for name, (parsed, low, high) in bounds.items():
        numeric[name] = parsed if np.isfinite(parsed) and low <= parsed <= high else np.nan

    property_type = "other"
    checks = [
        ("coop", ("co-op", "coop", "cooperative")),
        ("condo", ("condo", "condominium")),
        ("townhouse", ("townhouse", "townhome")),
        ("multifamily", ("multi-family", "multifamily", "duplex", "triplex", "fourplex")),
        ("land", ("vacant land", "buildable lot", "residential lot", "development site")),
        ("manufactured", ("manufactured home", "mobile home")),
        ("single_family", ("single family", "single-family")),
    ]
    for name, terms in checks:
        if any(term in lower for term in terms):
            property_type = name
            break

    numeric.update(
        {
            "ptype": property_type,
            "text_len": float(len(text)),
            "word_count": float(len(text.split())),
            "digit_count": float(sum(char.isdigit() for char in text)),
            "dollar_count": float(text.count("$")),
            "luxury_kw_count": float(
                sum(
                    term in lower
                    for term in (
                        "waterfront",
                        "oceanfront",
                        "beachfront",
                        "penthouse",
                        "estate",
                        "guest house",
                        "private dock",
                        "panoramic view",
                        "luxury",
                        "gated",
                        "acreage",
                        "central park",
                        "manhattan",
                    )
                )
            ),
        }
    )
    return numeric


def structured_tokens(row: pd.Series) -> str:
    tokens: list[str] = []
    if np.isfinite(row.beds):
        tokens.append(f"NUM_BED_{int(round(row.beds))}")
    if np.isfinite(row.baths):
        tokens.append(f"NUM_BATH_{round(row.baths * 2) / 2:g}")
    if np.isfinite(row.sqft):
        tokens.append(f"NUM_SQFT_{int(round(row.sqft / 250) * 250)}")
    if np.isfinite(row.acres):
        bins = "LT025" if row.acres < 0.25 else "025_05" if row.acres < 0.5 else "05_1" if row.acres < 1 else "1_2" if row.acres < 2 else "2_5" if row.acres < 5 else "GE5"
        tokens.append(f"NUM_ACRE_{bins}")
    if np.isfinite(row.garage):
        tokens.append(f"NUM_GARAGE_{int(round(row.garage))}")
    if np.isfinite(row.year):
        tokens.append(f"NUM_YEAR_{int(row.year // 10 * 10)}S")
    tokens.append(f"PROP_{str(row.ptype).upper()}")
    return " ".join(tokens)


def monotonic_band_probabilities(cumulative: np.ndarray) -> np.ndarray:
    cumulative = np.minimum.accumulate(np.clip(cumulative, 0.0, 1.0), axis=1)
    bands = np.column_stack(
        [1.0 - cumulative[:, 0], cumulative[:, :-1] - cumulative[:, 1:], cumulative[:, -1]]
    )
    bands = np.clip(bands, 0.0, None)
    return bands / np.maximum(bands.sum(axis=1, keepdims=True), 1e-9)


def prior_shift_probabilities(cumulative: np.ndarray, priors: np.ndarray, strength: float = 0.5) -> np.ndarray:
    clipped = np.clip(cumulative, 1e-6, 1.0 - 1e-6)
    logits = np.log(clipped / (1.0 - clipped))
    shift = np.log((1.0 - priors) / priors)
    return 1.0 / (1.0 + np.exp(-np.clip(logits + strength * shift, -30, 30)))


LOCATION_CUES = re.compile(
    r"\b(?:located|location|heart of|neighborhood|community|district|county|village|town|city|downtown|waterfront|near|minutes from|close to)\b",
    re.I,
)


def location_context(value: object) -> str:
    text = clean_text(value)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    kept = sentences[:1] + [sentence for sentence in sentences[1:] if LOCATION_CUES.search(sentence)]
    return (" ".join(text.split()[:45]) + " " + " ".join(kept[:4])).strip()


def aggregate_prior(matrix, prior: np.ndarray) -> np.ndarray:
    result = np.zeros((matrix.shape[0], 5), dtype=np.float32)
    counts = np.diff(matrix.indptr).astype(np.float32)
    result[:, 0] = np.asarray(matrix @ prior).ravel() / np.maximum(counts, 1)
    result[:, 4] = counts
    for row in range(matrix.shape[0]):
        indices = matrix.indices[matrix.indptr[row] : matrix.indptr[row + 1]]
        if not len(indices):
            continue
        values = prior[indices]
        result[row, 1] = float(values.max())
        result[row, 2] = float(values.min())
        result[row, 3] = float(np.mean(values if len(values) <= 3 else np.partition(values, -3)[-3:]))
    return result


def weighted_median(values: np.ndarray, weights: np.ndarray) -> float:
    order = np.argsort(values)
    sorted_values, sorted_weights = values[order], weights[order]
    return float(sorted_values[np.searchsorted(np.cumsum(sorted_weights), 0.5 * sorted_weights.sum())])


def retrieval_features(model: NearestNeighbors, queries, prices: np.ndarray) -> np.ndarray:
    distance, indices = model.kneighbors(queries, return_distance=True)
    similarities = np.clip(1.0 - distance, 0.0, 1.0)
    neighbor_prices = prices[indices]
    result = np.zeros((queries.shape[0], 7), dtype=np.float32)
    for row in range(queries.shape[0]):
        similarity = similarities[row]
        values = neighbor_prices[row].astype(float)
        weights = np.power(similarity + 1e-4, 4)
        result[row] = (
            similarity[0],
            similarity[0] - similarity[1],
            values[0],
            weighted_median(values, weights),
            np.median(values),
            np.median(np.abs(values - np.median(values))),
            np.average(values, weights=weights),
        )
    return result


def route_predictions(base: np.ndarray, cumulative: np.ndarray, experts: np.ndarray, medians: np.ndarray) -> dict[str, np.ndarray]:
    probs = monotonic_band_probabilities(cumulative)
    hard = probs.argmax(axis=1)
    median_band = (np.cumsum(probs, axis=1) < 0.5).sum(axis=1)
    rows = np.arange(len(base))
    expert_soft = np.sum(probs * experts, axis=1)
    expert_hard = experts[rows, hard]
    expert_median = experts[rows, median_band]
    if medians.ndim == 1:
        band_soft = probs @ medians
        band_median = medians[median_band]
    else:
        band_soft = np.sum(probs * medians, axis=1)
        band_median = medians[rows, median_band]
    tail_gate = np.clip(cumulative[:, 5], 0.0, 1.0)
    tail_guard = base * (1.0 - tail_gate) + expert_soft * tail_gate
    return {
        "base": base,
        "band_soft": band_soft,
        "band_median": band_median,
        "expert_soft": expert_soft,
        "expert_hard": expert_hard,
        "expert_median": expert_median,
        "tail_guard": tail_guard,
        "blend_base_soft_25": 0.75 * base + 0.25 * expert_soft,
        "blend_base_soft_50": 0.50 * base + 0.50 * expert_soft,
        "blend_base_median_25": 0.75 * base + 0.25 * expert_median,
    }


def band_report(y: np.ndarray, pred: np.ndarray) -> list[dict[str, float | int | str]]:
    rows = []
    for low, high in zip(EDGES[:-1], EDGES[1:]):
        mask = (y >= low) & (y < high)
        rows.append(
            {
                "band": f"{low:,.0f}-{high:,.0f}",
                "n": int(mask.sum()),
                "mae": float(mean_absolute_error(y[mask], pred[mask])),
                "bias": float(np.mean(pred[mask] - y[mask])),
            }
        )
    return rows


In [ ]:
from types import SimpleNamespace
from glob import glob
import os

SEED = 42
N_FOLDS = 5
WORD_FEATURES = 180_000
CHAR_FEATURES = 140_000

def locate_data_dir() -> Path:
    override = os.environ.get("DATA_DIR")
    if override and (Path(override) / "train.csv").exists():
        return Path(override)
    patterns = [
        "/kaggle/input/competitions/*/train.csv",
        "/kaggle/input/*/train.csv",
        "/kaggle/input/*/*/train.csv",
        "/mnt/data/holomine_task2/train.csv",
        "train.csv",
    ]
    for pattern in patterns:
        hits = glob(pattern)
        for hit in hits:
            candidate = Path(hit).parent
            if (candidate / "test.csv").exists() and (candidate / "sample_submission.csv").exists():
                return candidate
    raise FileNotFoundError("train.csv/test.csv/sample_submission.csv tidak ditemukan")

DATA_DIR = locate_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
args = SimpleNamespace(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    folds=N_FOLDS,
    word_features=WORD_FEATURES,
    char_features=CHAR_FEATURES,
)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 1. Data, parser, dan representasi sparse

Teks dibersihkan tanpa membuang angka atau istilah properti. Parser mengekstrak kamar tidur, kamar mandi, luas bangunan, luas tanah, garasi, tahun bangun, jumlah lantai, dan tipe properti. Word n-gram menangkap istilah serta nama lokasi, sedangkan character n-gram menangkap variasi ejaan, singkatan, dan pola teks yang jarang.


In [ ]:
args.output_dir.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(args.data_dir / "train.csv")
test = pd.read_csv(args.data_dir / "test.csv")
y = train.listPrice.to_numpy(dtype=float)
n_train = len(train)
all_text = pd.concat([train.text, test.text], ignore_index=True).map(clean_text)
structured = pd.DataFrame([parse_property(text) for text in all_text])
augmented = pd.Series(
    [f"{structured_tokens(structured.iloc[i])} SEP {all_text.iloc[i]}" for i in range(len(all_text))]
)
coverage = {
    col: float(structured.loc[: n_train - 1, col].notna().mean())
    for col in ("beds", "baths", "sqft", "acres", "garage", "year")
}
print("Parser coverage:", coverage, flush=True)

word = TfidfVectorizer(
    ngram_range=(1, 2), min_df=3, max_features=args.word_features,
    sublinear_tf=True, strip_accents="unicode", dtype=np.float32,
)
char = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=5,
    max_features=args.char_features, sublinear_tf=True, dtype=np.float32,
)
x_word = word.fit_transform(augmented)
x_char = char.fit_transform(augmented)
x_all = hstack([x_word, x_char], format="csr", dtype=np.float32)
del x_word, x_char
x_train, x_test = x_all[:n_train], x_all[n_train:]
print("TF-IDF shape:", x_all.shape, flush=True)

strata = pd.qcut(np.log1p(y), q=20, labels=False, duplicates="drop")
splitter = StratifiedKFold(args.folds, shuffle=True, random_state=SEED)
fold_indices = list(splitter.split(np.zeros(n_train), strata))


## 2. Sparse backbone, ordinal classifiers, dan specialist models

Backbone menggabungkan beberapa regresi sparse yang dilatih pada target asli dan transformasi logaritmik. Classifier ordinal memperkirakan peluang harga melewati setiap batas. Specialist models dilatih pada rentang yang saling tumpang tindih agar observasi dekat batas harga tidak berpindah model secara terlalu tajam.

Seluruh fitur supervised untuk baris train dibuat secara cross-fitted.


In [ ]:
base_oof_parts = np.zeros((n_train, 3), dtype=float)
base_test_parts = np.zeros((len(test), 3), dtype=float)
cumulative_oof = np.zeros((n_train, len(THRESHOLDS)), dtype=float)
cumulative_test = np.zeros((len(test), len(THRESHOLDS)), dtype=float)
experts_oof = np.zeros((n_train, len(EDGES) - 1), dtype=float)
experts_test = np.zeros((len(test), len(EDGES) - 1), dtype=float)
fold_medians_oof = np.zeros((n_train, len(EDGES) - 1), dtype=float)
test_medians = np.zeros(len(EDGES) - 1, dtype=float)

for fold, (fit_idx, val_idx) in enumerate(fold_indices):
    print(f"Fold {fold + 1}/{args.folds}", flush=True)
    transforms = [
        (np.log1p, lambda z: np.expm1(np.clip(z, 0, 25)), 0.7),
        (lambda a: np.power(a / 1_000_000.0, 0.20), lambda z: np.power(np.clip(z, 0, 10), 5) * 1_000_000.0, 0.30),
        (lambda a: np.power(a / 1_000_000.0, 0.25), lambda z: np.power(np.clip(z, 0, 10), 4) * 1_000_000.0, 0.35),
    ]
    for column, (transform, inverse, alpha) in enumerate(transforms):
        model = Ridge(alpha=alpha, solver="lsqr", tol=1e-3)
        model.fit(x_train[fit_idx], transform(y[fit_idx]))
        base_oof_parts[val_idx, column] = inverse(model.predict(x_train[val_idx]))
        base_test_parts[:, column] += inverse(model.predict(x_test)) / args.folds

    for threshold_index, threshold in enumerate(THRESHOLDS):
        label = (y[fit_idx] >= threshold).astype(int)
        classifier = SGDClassifier(
            loss="log_loss", alpha=1.5e-5, penalty="l2", max_iter=2_000,
            tol=1e-4, average=True, random_state=SEED + 97 * threshold_index + fold,
        )
        classifier.fit(x_train[fit_idx], label)
        cumulative_oof[val_idx, threshold_index] = classifier.predict_proba(x_train[val_idx])[:, 1]
        cumulative_test[:, threshold_index] += classifier.predict_proba(x_test)[:, 1] / args.folds

    fit_band = np.digitize(y[fit_idx], THRESHOLDS, right=False)
    for band in range(len(EDGES) - 1):
        distance = np.abs(fit_band - band)
        weights = np.where(distance == 0, 1.0, np.where(distance == 1, 0.30, np.where(distance == 2, 0.06, 0.015)))
        alpha = 1.2 if band < 6 else 2.5
        expert = Ridge(alpha=alpha, solver="lsqr", tol=1e-3)
        expert.fit(x_train[fit_idx], np.log1p(y[fit_idx]), sample_weight=weights)
        val_prediction = np.expm1(np.clip(expert.predict(x_train[val_idx]), 0, 25))
        test_prediction = np.expm1(np.clip(expert.predict(x_test), 0, 25))
        lower, upper = EDGES[band], EDGES[band + 1]
        clip_low = max(1_000.0, lower)
        clip_high = 50_000_000.0 if not np.isfinite(upper) else upper
        experts_oof[val_idx, band] = np.clip(val_prediction, clip_low, clip_high)
        experts_test[:, band] += np.clip(test_prediction, clip_low, clip_high) / args.folds
        in_band = fit_band == band
        median = float(np.median(y[fit_idx][in_band]))
        fold_medians_oof[val_idx, band] = median
        test_medians[band] += median / args.folds

base_oof = base_oof_parts[:, 1]
base_test = base_test_parts[:, 1]
oof_routes = route_predictions(base_oof, cumulative_oof, experts_oof, fold_medians_oof)
test_routes = route_predictions(base_test, cumulative_test, experts_test, test_medians)


## 3. Prior correction, routing, retrieval, dan residual model

Probabilitas ordinal dikalibrasi dan diubah menjadi peluang tiap rentang harga. Prediksi specialist digabungkan menggunakan soft routing. Token lokasi, tetangga TF-IDF, dan structured features menambah informasi yang tidak tertangkap regresi global. LightGBM terakhir mempelajari residual dari prediksi sparse yang sudah terbentuk.


In [ ]:
# Train-only prior correction: class rarity is softened, not removed.  This
# makes the excellent tail ranking useful without globally inflating prices.
priors = np.array([(y >= threshold).mean() for threshold in THRESHOLDS])
shifted_oof = prior_shift_probabilities(cumulative_oof, priors, strength=0.5)
shifted_test = prior_shift_probabilities(cumulative_test, priors, strength=0.5)
shifted_oof_probs = monotonic_band_probabilities(shifted_oof)
shifted_test_probs = monotonic_band_probabilities(shifted_test)
oof_median_band = (np.cumsum(shifted_oof_probs, axis=1) < 0.5).sum(axis=1)
test_median_band = (np.cumsum(shifted_test_probs, axis=1) < 0.5).sum(axis=1)
shifted_oof_expert = experts_oof[np.arange(n_train), oof_median_band]
shifted_test_expert = experts_test[np.arange(len(test)), test_median_band]
routed_oof = 0.40 * base_oof + 0.60 * shifted_oof_expert
routed_test = 0.40 * base_test + 0.60 * shifted_test_expert
oof_routes["prior_corrected_route"] = routed_oof
test_routes["prior_corrected_route"] = routed_test

# Location-token priors and high-similarity retrieval are compact numerical
# evidence for the residual model.  Both are cross-fitted.
location_text = all_text.map(location_context)
location_vectorizer = CountVectorizer(
    ngram_range=(1, 3), min_df=4, max_features=50_000,
    binary=True, strip_accents="unicode", dtype=np.float32,
)
x_location = location_vectorizer.fit_transform(location_text).tocsr().astype(np.float32)
x_location_train, x_location_test = x_location[:n_train], x_location[n_train:]
prior_oof = np.zeros((n_train, 5), dtype=np.float32)
prior_test = np.zeros((len(test), 5), dtype=np.float32)
global_log_mean = float(np.log1p(y).mean())
for fit_idx, val_idx in fold_indices:
    counts = np.asarray(x_location_train[fit_idx].sum(axis=0)).ravel().astype(float)
    sums = np.asarray(x_location_train[fit_idx].T @ np.log1p(y[fit_idx])).ravel().astype(float)
    token_prior = ((sums + 20.0 * global_log_mean) / (counts + 20.0)).astype(np.float32)
    prior_oof[val_idx] = aggregate_prior(x_location_train[val_idx], token_prior)
    prior_test += aggregate_prior(x_location_test, token_prior) / args.folds

retrieval_oof = np.zeros((n_train, 7), dtype=np.float32)
retrieval_test = np.zeros((len(test), 7), dtype=np.float32)
for fold, (fit_idx, val_idx) in enumerate(fold_indices):
    neighbors = NearestNeighbors(n_neighbors=6, metric="cosine", algorithm="brute", n_jobs=-1)
    neighbors.fit(x_train[fit_idx])
    retrieval_oof[val_idx] = retrieval_features(neighbors, x_train[val_idx], y[fit_idx])
    retrieval_test += retrieval_features(neighbors, x_test, y[fit_idx]) / args.folds
    print(f"Retrieval fold {fold + 1}/{args.folds}", flush=True)

numeric_columns = [
    "beds", "baths", "sqft", "acres", "lot_sqft", "garage", "year", "stories",
    "text_len", "word_count", "digit_count", "dollar_count", "luxury_kw_count",
]
structured_numeric = structured[numeric_columns].apply(pd.to_numeric, errors="coerce")
train_medians = structured_numeric.iloc[:n_train].median()
structured_numeric = structured_numeric.fillna(train_medians).fillna(0.0)
property_dummies = pd.get_dummies(structured.ptype, prefix="ptype", dtype=np.float32)
structured_matrix = np.column_stack([structured_numeric.to_numpy(np.float32), property_dummies.to_numpy(np.float32)])

meta_oof = np.column_stack(
    [
        base_oof_parts,
        base_oof,
        np.log1p(base_oof),
        base_oof_parts[:, 2] - base_oof_parts[:, 0],
        cumulative_oof,
        monotonic_band_probabilities(cumulative_oof),
        shifted_oof_probs,
        experts_oof,
        routed_oof,
        prior_oof,
        retrieval_oof,
        structured_matrix[:n_train],
    ]
).astype(np.float32)
meta_test = np.column_stack(
    [
        base_test_parts,
        base_test,
        np.log1p(base_test),
        base_test_parts[:, 2] - base_test_parts[:, 0],
        cumulative_test,
        monotonic_band_probabilities(cumulative_test),
        shifted_test_probs,
        experts_test,
        routed_test,
        prior_test,
        retrieval_test,
        structured_matrix[n_train:],
    ]
).astype(np.float32)
print("Meta shape:", meta_oof.shape, flush=True)

meta_params = {
    "objective": "l1", "metric": "l1", "learning_rate": 0.025,
    "num_leaves": 31, "min_data_in_leaf": 45, "feature_fraction": 0.85,
    "bagging_fraction": 0.85, "bagging_freq": 1, "lambda_l1": 0.2,
    "lambda_l2": 3.0, "max_bin": 127, "force_col_wise": True,
    "verbosity": -1, "seed": SEED, "num_threads": -1,
}
meta_weights = np.ones(n_train, dtype=np.float32)
meta_weights *= np.where(y >= np.quantile(y, 0.95), 1.20, 1.0)
meta_weights *= np.where(y >= np.quantile(y, 0.99), 1.25, 1.0)
for base_name, meta_base_oof, meta_base_test in (
    ("meta_from_ridge", base_oof, base_test),
    ("meta_from_route", routed_oof, routed_test),
):
    corrected_oof = np.zeros(n_train, dtype=float)
    corrected_test = np.zeros(len(test), dtype=float)
    residual = y - meta_base_oof
    for fold, (fit_idx, val_idx) in enumerate(fold_indices):
        fit_data = lgb.Dataset(meta_oof[fit_idx], label=residual[fit_idx], weight=meta_weights[fit_idx])
        val_data = lgb.Dataset(meta_oof[val_idx], label=residual[val_idx], reference=fit_data)
        model = lgb.train(
            meta_params, fit_data, num_boost_round=1_800, valid_sets=[val_data],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
        )
        corrected_oof[val_idx] = meta_base_oof[val_idx] + model.predict(meta_oof[val_idx], num_iteration=model.best_iteration)
        corrected_test += (meta_base_test + model.predict(meta_test, num_iteration=model.best_iteration)) / args.folds
        print(base_name, "fold", fold + 1, "best", model.best_iteration, flush=True)
    oof_routes[base_name] = np.clip(corrected_oof, 1_000, 50_000_000)
    test_routes[base_name] = np.clip(corrected_test, 1_000, 50_000_000)


## 4. Latent-domain dan price-regime experts

Campuran tipe pasar pada train dan test dapat berbeda. Domain internal dibentuk dari petunjuk geografis yang sudah ada di kolom teks. Sparse classifier kemudian menghasilkan probabilitas domain untuk seluruh baris. Probabilitas ini digunakan sebagai bobot lunak, bukan sebagai kelas yang mutlak.

Bagian ini menambahkan:

- regresi harga berbobot domain;
- classifier ordinal berbobot domain;
- specialist untuk kombinasi domain dan rentang harga;
- prior token lokasi per rentang harga;
- nearest-neighbor retrieval di dalam domain;
- estimasi perubahan proporsi rentang harga;
- residual LightGBM dengan bobot sesuai estimasi komposisi test.

Semua komponen supervised tetap cross-fitted.


In [ ]:
# ============================================================
# V6 INTERNAL-ONLY DOMAIN × REGIME EXTENSION
# ============================================================

NY_ANCHORS = [
    "new york", "nyc", "manhattan", "brooklyn", "queens", "bronx",
    "staten island", "long island", "hamptons", "westchester", "hudson valley",
    "albany", "buffalo", "rochester", "syracuse", "yonkers", "central park",
    "upper east side", "upper west side", "soho", "tribeca", "harlem",
    "chelsea", "greenwich village", "astoria", "flushing", "adirondack",
    "saratoga", "hudson river", "finger lakes",
]

OREGON_ANCHORS = [
    "oregon", "portland", "bend", "eugene", "salem", "beaverton", "gresham",
    "hillsboro", "lake oswego", "medford", "grants pass", "redmond",
    "willamette", "deschutes", "clackamas", "multnomah", "hood river",
    "central oregon", "southern oregon", "coos bay", "corvallis", "ashland",
    "roseburg", "tualatin", "tigard", "oregon coast", "cascades",
]

ny_pattern = re.compile("|".join(map(re.escape, NY_ANCHORS)), re.I)
oregon_pattern = re.compile("|".join(map(re.escape, OREGON_ANCHORS)), re.I)
ny_anchor = all_text.str.contains(ny_pattern).to_numpy()
oregon_anchor = all_text.str.contains(oregon_pattern).to_numpy()
anchor_mask = ny_anchor ^ oregon_anchor
anchor_label = ny_anchor[anchor_mask].astype(int)

domain_classifier = SGDClassifier(
    loss="log_loss", alpha=3e-6, penalty="l2", max_iter=3_000,
    tol=1e-5, average=True, class_weight="balanced", random_state=SEED + 600,
)
domain_classifier.fit(x_all[anchor_mask], anchor_label)
domain_probability_all = domain_classifier.predict_proba(x_all)[:, 1].astype(np.float32)
domain_probability_train = domain_probability_all[:n_train]
domain_probability_test = domain_probability_all[n_train:]

print("Domain anchors:", int(anchor_mask.sum()), "/", len(all_text))
print(
    "Mean NY-like probability train/test:",
    f"{domain_probability_train.mean():.3f}",
    f"{domain_probability_test.mean():.3f}",
)

# Sharpen only highly confident domain probabilities; ambiguous rows remain soft.
domain_sharpness = 1.5
domain_odds_train = np.power(np.clip(domain_probability_train, 1e-5, 1 - 1e-5), domain_sharpness)
domain_reverse_train = np.power(1.0 - np.clip(domain_probability_train, 1e-5, 1 - 1e-5), domain_sharpness)
domain_route_train = domain_odds_train / (domain_odds_train + domain_reverse_train)
domain_odds_test = np.power(np.clip(domain_probability_test, 1e-5, 1 - 1e-5), domain_sharpness)
domain_reverse_test = np.power(1.0 - np.clip(domain_probability_test, 1e-5, 1 - 1e-5), domain_sharpness)
domain_route_test = domain_odds_test / (domain_odds_test + domain_reverse_test)

domain_base_oof = np.zeros(n_train, dtype=np.float64)
domain_base_test = np.zeros(len(test), dtype=np.float64)
domain_ordinal_oof = np.zeros_like(cumulative_oof, dtype=np.float32)
domain_ordinal_test = np.zeros_like(cumulative_test, dtype=np.float32)
domain_experts_oof = np.zeros_like(experts_oof, dtype=np.float64)
domain_experts_test = np.zeros_like(experts_test, dtype=np.float64)

domain_floor = 0.05
for fold, (fit_idx, val_idx) in enumerate(fold_indices):
    transformed_target = np.power(y[fit_idx] / 1_000_000.0, 0.20)
    ny_weight = domain_floor + (1.0 - domain_floor) * domain_route_train[fit_idx]
    oregon_weight = domain_floor + (1.0 - domain_floor) * (1.0 - domain_route_train[fit_idx])

    ny_model = Ridge(alpha=0.50, solver="lsqr", tol=1e-3)
    oregon_model = Ridge(alpha=0.50, solver="lsqr", tol=1e-3)
    ny_model.fit(x_train[fit_idx], transformed_target, sample_weight=ny_weight)
    oregon_model.fit(x_train[fit_idx], transformed_target, sample_weight=oregon_weight)

    ny_val = np.power(np.clip(ny_model.predict(x_train[val_idx]), 0, 10), 5) * 1_000_000.0
    oregon_val = np.power(np.clip(oregon_model.predict(x_train[val_idx]), 0, 10), 5) * 1_000_000.0
    domain_base_oof[val_idx] = (
        domain_route_train[val_idx] * ny_val
        + (1.0 - domain_route_train[val_idx]) * oregon_val
    )
    ny_test = np.power(np.clip(ny_model.predict(x_test), 0, 10), 5) * 1_000_000.0
    oregon_test = np.power(np.clip(oregon_model.predict(x_test), 0, 10), 5) * 1_000_000.0
    domain_base_test += (
        domain_route_test * ny_test + (1.0 - domain_route_test) * oregon_test
    ) / args.folds

    # Domain-conditional ordinal models. Weighted class balancing affects rank,
    # while final calibration/routing is handled downstream.
    for threshold_index, threshold in enumerate(THRESHOLDS):
        binary_target = (y[fit_idx] >= threshold).astype(int)
        domain_predictions_val = []
        domain_predictions_test = []
        for domain_weight, seed_offset in ((ny_weight, 0), (oregon_weight, 10_000)):
            positive_mass = max(float(domain_weight[binary_target == 1].sum()), 1.0)
            negative_mass = max(float(domain_weight[binary_target == 0].sum()), 1.0)
            balanced_weight = domain_weight.copy()
            balanced_weight[binary_target == 1] *= 0.5 * domain_weight.sum() / positive_mass
            balanced_weight[binary_target == 0] *= 0.5 * domain_weight.sum() / negative_mass
            classifier = SGDClassifier(
                loss="log_loss", alpha=1.5e-5, penalty="l2", max_iter=2_000,
                tol=1e-4, average=True,
                random_state=SEED + seed_offset + threshold_index * 97 + fold,
            )
            classifier.fit(x_train[fit_idx], binary_target, sample_weight=balanced_weight)
            domain_predictions_val.append(classifier.predict_proba(x_train[val_idx])[:, 1])
            domain_predictions_test.append(classifier.predict_proba(x_test)[:, 1])
        domain_ordinal_oof[val_idx, threshold_index] = (
            domain_route_train[val_idx] * domain_predictions_val[0]
            + (1.0 - domain_route_train[val_idx]) * domain_predictions_val[1]
        )
        domain_ordinal_test[:, threshold_index] += (
            domain_route_test * domain_predictions_test[0]
            + (1.0 - domain_route_test) * domain_predictions_test[1]
        ) / args.folds

    fit_band = np.digitize(y[fit_idx], THRESHOLDS, right=False)
    for band in range(len(EDGES) - 1):
        band_distance = np.abs(fit_band - band)
        band_weight = np.where(
            band_distance == 0, 1.0,
            np.where(band_distance == 1, 0.30, np.where(band_distance == 2, 0.06, 0.015)),
        )
        domain_band_val = []
        domain_band_test = []
        for domain_weight in (ny_weight, oregon_weight):
            expert_weight = domain_weight * band_weight
            expert = Ridge(
                alpha=1.5 if band < 6 else 3.0,
                solver="lsqr", tol=1e-3,
            )
            expert.fit(x_train[fit_idx], np.log1p(y[fit_idx]), sample_weight=expert_weight)
            domain_band_val.append(np.expm1(np.clip(expert.predict(x_train[val_idx]), 0, 25)))
            domain_band_test.append(np.expm1(np.clip(expert.predict(x_test), 0, 25)))

        lower, upper = EDGES[band], EDGES[band + 1]
        upper_clip = 50_000_000.0 if not np.isfinite(upper) else upper
        mixed_val = (
            domain_route_train[val_idx] * domain_band_val[0]
            + (1.0 - domain_route_train[val_idx]) * domain_band_val[1]
        )
        mixed_test = (
            domain_route_test * domain_band_test[0]
            + (1.0 - domain_route_test) * domain_band_test[1]
        )
        domain_experts_oof[val_idx, band] = np.clip(mixed_val, max(1_000.0, lower), upper_clip)
        domain_experts_test[:, band] += np.clip(mixed_test, max(1_000.0, lower), upper_clip) / args.folds

    print(f"V6 domain models fold {fold + 1}/{args.folds}")

print("Domain base OOF MAE:", f"{mean_absolute_error(y, domain_base_oof):,.0f}")
for threshold_index, threshold in enumerate(THRESHOLDS):
    print(
        f"Domain ordinal >= {threshold:,.0f} AUC",
        f"{roc_auc_score(y >= threshold, domain_ordinal_oof[:, threshold_index]):.4f}",
    )

# Cross-fitted location-regime priors: each location token gets a smoothed
# probability for every cumulative price threshold.
location_regime_oof = np.zeros((n_train, len(THRESHOLDS) * 4), dtype=np.float32)
location_regime_test = np.zeros((len(test), len(THRESHOLDS) * 4), dtype=np.float32)
for fit_idx, val_idx in fold_indices:
    token_count = np.asarray(x_location_train[fit_idx].sum(axis=0)).ravel().astype(np.float64)
    for threshold_index, threshold in enumerate(THRESHOLDS):
        label = (y[fit_idx] >= threshold).astype(np.float64)
        global_rate = float(label.mean())
        positive_count = np.asarray(x_location_train[fit_idx].T @ label).ravel().astype(np.float64)
        token_rate = ((positive_count + 30.0 * global_rate) / (token_count + 30.0)).astype(np.float32)
        val_aggregate = aggregate_prior(x_location_train[val_idx], token_rate)[:, :4]
        test_aggregate = aggregate_prior(x_location_test, token_rate)[:, :4]
        start = threshold_index * 4
        location_regime_oof[val_idx, start:start + 4] = val_aggregate
        location_regime_test[:, start:start + 4] += test_aggregate / args.folds

# Retrieval inside the inferred domain. This avoids comparing Manhattan luxury
# listings with similarly worded Oregon properties.
domain_retrieval_oof = np.zeros_like(retrieval_oof)
domain_retrieval_test = np.zeros_like(retrieval_test)
for fold, (fit_idx, val_idx) in enumerate(fold_indices):
    val_domain_features = []
    test_domain_features = []
    for is_ny in (True, False):
        if is_ny:
            subset = fit_idx[domain_route_train[fit_idx] >= 0.5]
        else:
            subset = fit_idx[domain_route_train[fit_idx] < 0.5]
        neighbors = NearestNeighbors(n_neighbors=6, metric="cosine", algorithm="brute", n_jobs=-1)
        neighbors.fit(x_train[subset])
        val_domain_features.append(retrieval_features(neighbors, x_train[val_idx], y[subset]))
        test_domain_features.append(retrieval_features(neighbors, x_test, y[subset]))
    domain_retrieval_oof[val_idx] = (
        domain_route_train[val_idx, None] * val_domain_features[0]
        + (1.0 - domain_route_train[val_idx, None]) * val_domain_features[1]
    )
    domain_retrieval_test += (
        domain_route_test[:, None] * test_domain_features[0]
        + (1.0 - domain_route_test[:, None]) * test_domain_features[1]
    ) / args.folds
    print(f"V6 domain retrieval fold {fold + 1}/{args.folds}")

# Estimate cumulative test price proportions from OOF score distributions.
# This is Black-Box Shift Estimation and uses no test labels.
estimated_test_cumulative = []
for threshold_index, threshold in enumerate(THRESHOLDS):
    positive = y >= threshold
    positive_mean = float(cumulative_oof[positive, threshold_index].mean())
    negative_mean = float(cumulative_oof[~positive, threshold_index].mean())
    test_mean = float(cumulative_test[:, threshold_index].mean())
    estimate = (test_mean - negative_mean) / max(positive_mean - negative_mean, 1e-6)
    estimated_test_cumulative.append(float(np.clip(estimate, 0.001, 0.999)))
estimated_test_cumulative = np.minimum.accumulate(np.asarray(estimated_test_cumulative))
estimated_test_band_share = np.r_[
    1.0 - estimated_test_cumulative[0],
    estimated_test_cumulative[:-1] - estimated_test_cumulative[1:],
    estimated_test_cumulative[-1],
]
estimated_test_band_share = np.clip(estimated_test_band_share, 1e-4, None)
estimated_test_band_share /= estimated_test_band_share.sum()
train_band = np.digitize(y, THRESHOLDS, right=False)
train_band_share = np.bincount(train_band, minlength=len(EDGES) - 1) / len(train_band)
label_shift_weight = estimated_test_band_share[train_band] / np.maximum(train_band_share[train_band], 1e-6)
train_ny_share = float(domain_probability_train.mean())
test_ny_share = float(domain_probability_test.mean())
domain_shift_weight = (
    domain_probability_train * (test_ny_share / max(train_ny_share, 1e-6))
    + (1.0 - domain_probability_train)
    * ((1.0 - test_ny_share) / max(1.0 - train_ny_share, 1e-6))
)
label_shift_weight *= domain_shift_weight
label_shift_weight = np.clip(label_shift_weight, 0.40, 4.00).astype(np.float32)

print("Estimated test cumulative shares:", dict(zip(THRESHOLDS.astype(int), estimated_test_cumulative.round(4))))
print("Estimated test band shares:", estimated_test_band_share.round(4))
print("Estimated NY-like share train/test:", round(train_ny_share, 4), round(test_ny_share, 4))

domain_band_oof = monotonic_band_probabilities(domain_ordinal_oof)
domain_band_test = monotonic_band_probabilities(domain_ordinal_test)
domain_soft_expert_oof = np.sum(domain_band_oof * domain_experts_oof, axis=1)
domain_soft_expert_test = np.sum(domain_band_test * domain_experts_test, axis=1)

v6_meta_oof = np.column_stack([
    meta_oof,
    domain_probability_train,
    domain_base_oof,
    domain_base_oof - base_oof,
    domain_ordinal_oof,
    domain_band_oof,
    domain_experts_oof,
    domain_soft_expert_oof,
    location_regime_oof,
    domain_retrieval_oof,
]).astype(np.float32)

v6_meta_test = np.column_stack([
    meta_test,
    domain_probability_test,
    domain_base_test,
    domain_base_test - base_test,
    domain_ordinal_test,
    domain_band_test,
    domain_experts_test,
    domain_soft_expert_test,
    location_regime_test,
    domain_retrieval_test,
]).astype(np.float32)

print("V6 meta shape:", v6_meta_oof.shape, v6_meta_test.shape)

v6_candidates_oof = {}
v6_candidates_test = {}
for candidate_name, candidate_base_oof, candidate_base_test in (
    ("v6_global_base", base_oof, base_test),
    ("v6_domain_base", domain_base_oof, domain_base_test),
):
    residual = y - candidate_base_oof
    candidate_oof = np.zeros(n_train, dtype=np.float64)
    candidate_test = np.zeros(len(test), dtype=np.float64)
    for fold, (fit_idx, val_idx) in enumerate(fold_indices):
        fit_data = lgb.Dataset(
            v6_meta_oof[fit_idx], label=residual[fit_idx],
            weight=label_shift_weight[fit_idx],
        )
        val_data = lgb.Dataset(v6_meta_oof[val_idx], label=residual[val_idx], reference=fit_data)
        model = lgb.train(
            meta_params, fit_data, num_boost_round=2_200, valid_sets=[val_data],
            callbacks=[lgb.early_stopping(120, verbose=False), lgb.log_evaluation(0)],
        )
        candidate_oof[val_idx] = candidate_base_oof[val_idx] + model.predict(
            v6_meta_oof[val_idx], num_iteration=model.best_iteration,
        )
        candidate_test += (
            candidate_base_test + model.predict(v6_meta_test, num_iteration=model.best_iteration)
        ) / args.folds
        print(candidate_name, "fold", fold + 1, "best", model.best_iteration)
    v6_candidates_oof[candidate_name] = np.clip(candidate_oof, 1_000, 50_000_000)
    v6_candidates_test[candidate_name] = np.clip(candidate_test, 1_000, 50_000_000)

for name in v6_candidates_oof:
    oof_routes[name] = v6_candidates_oof[name]
    test_routes[name] = v6_candidates_test[name]

v6_selection_pool_oof = {"meta_from_ridge": oof_routes["meta_from_ridge"], **v6_candidates_oof}
v6_selection_pool_test = {"meta_from_ridge": test_routes["meta_from_ridge"], **v6_candidates_test}
v6_shift_scores = {
    name: mean_absolute_error(y, prediction, sample_weight=label_shift_weight)
    for name, prediction in v6_selection_pool_oof.items()
}
v6_selected = min(v6_shift_scores, key=v6_shift_scores.get)
print("V6 ordinary OOF:", {name: round(mean_absolute_error(y, pred)) for name, pred in v6_candidates_oof.items()})
print("V6 estimated-test-mix OOF:", {name: round(score) for name, score in v6_shift_scores.items()})
print("V6 selected:", v6_selected)


## 5. Evaluasi dan pembuatan submission

Cell terakhir merangkum AUC ordinal, MAE setiap kandidat, laporan error per rentang harga, dan distribusi prediksi test. Artefak yang dihasilkan adalah `submission_v6.csv`, `oof_v6.csv`, dan `report_v6.json`.


In [ ]:
auc = {
    f"ge_{int(threshold)}": float(roc_auc_score(y >= threshold, cumulative_oof[:, j]))
    for j, threshold in enumerate(THRESHOLDS)
}
scores = {name: float(mean_absolute_error(y, pred)) for name, pred in oof_routes.items()}
selected = v6_selected if "v6_selected" in globals() else min(scores, key=scores.get)
print("AUC:", auc, flush=True)
print("OOF MAE:", dict(sorted(scores.items(), key=lambda item: item[1])), flush=True)
print("Selected using estimated test-mix OOF:", selected, flush=True)

pd.DataFrame(
    {
        "id": train.id,
        "actual": y,
        **{name: pred for name, pred in oof_routes.items()},
    }
).to_csv(args.output_dir / "oof_v6.csv", index=False)
pd.DataFrame(
    {"id": test.id, "listPrice": np.clip(test_routes[selected], 1_000, 50_000_000)}
).to_csv(args.output_dir / "submission_v6.csv", index=False)
np.savez_compressed(
    args.output_dir / "routing_arrays_v6.npz",
    cumulative_oof=cumulative_oof,
    cumulative_test=cumulative_test,
    experts_oof=experts_oof,
    experts_test=experts_test,
    base_oof_parts=base_oof_parts,
    base_test_parts=base_test_parts,
    prior_oof=prior_oof,
    prior_test=prior_test,
    retrieval_oof=retrieval_oof,
    retrieval_test=retrieval_test,
)
report = {
    "parser_coverage": coverage,
    "ordinal_auc": auc,
    "oof_mae": scores,
    "selected_candidate": selected,
        "estimated_test_band_share": estimated_test_band_share.tolist() if "estimated_test_band_share" in globals() else None,
        "v6_shift_weighted_oof": v6_shift_scores if "v6_shift_scores" in globals() else None,
    "selected_band_report": band_report(y, oof_routes[selected]),
    "base_band_report": band_report(y, base_oof),
    "prediction_quantiles": pd.Series(test_routes[selected]).describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.97, 0.99]).to_dict(),
}
(args.output_dir / "report_v6.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
